In [15]:
import os
import time
from pathlib import Path

os.environ.setdefault("PYICEBERG_MAX_WORKERS", "32")

from dotenv import load_dotenv
from pyiceberg.catalog.rest import RestCatalog

load_dotenv(Path.cwd() / ".env") or load_dotenv(Path.cwd().parent / ".env")

True

In [16]:
ACCOUNT_ID = os.getenv("CLOUDFLARE_ACCOUNT_ID")
API_TOKEN = os.getenv("CLOUDFLARE_API_TOKEN")
BUCKET = os.getenv("R2_BUCKET_NAME", "market-data")

if not all([ACCOUNT_ID, API_TOKEN, BUCKET]):
    raise ValueError("Set CLOUDFLARE_ACCOUNT_ID, CLOUDFLARE_API_TOKEN, and R2_BUCKET_NAME")

catalog = RestCatalog(
    name=os.getenv("R2_DATA_CATALOG_NAME", "cloudflare_r2"),
    warehouse=os.getenv("R2_DATA_CATALOG_WAREHOUSE") or f"{ACCOUNT_ID}_{BUCKET}",
    uri=os.getenv("R2_DATA_CATALOG_URI") or f"https://catalog.cloudflarestorage.com/{ACCOUNT_ID}/{BUCKET}",
    token=API_TOKEN,
)

table = catalog.load_table(("cash", "ohlcv_by_symbol"))
table.spec(), table.sort_order()

(PartitionSpec(PartitionField(source_id=3, field_id=1000, transform=IdentityTransform(), name='nse_symbol'), PartitionField(source_id=2, field_id=1001, transform=YearTransform(), name='year_trade_date'), spec_id=0),
 SortOrder(SortField(source_id=3, transform=IdentityTransform(), direction=SortDirection.ASC, null_order=NullOrder.NULLS_LAST), SortField(source_id=2, transform=IdentityTransform(), direction=SortDirection.ASC, null_order=NullOrder.NULLS_LAST), SortField(source_id=1, transform=IdentityTransform(), direction=SortDirection.ASC, null_order=NullOrder.NULLS_LAST), order_id=1))

In [17]:
SYMBOL = "RELIANCE"
FROM_DATE = "2016-01-01"
TO_DATE = None
COLUMNS = ("datetime", "trade_date", "nse_symbol", "open", "high", "low", "close", "volume")

date_filter = f"trade_date >= '{FROM_DATE}'"
if TO_DATE:
    date_filter += f" AND trade_date <= '{TO_DATE}'"

row_filter = f"{date_filter} AND nse_symbol == '{SYMBOL}'"
row_filter

"trade_date >= '2016-01-01' AND nse_symbol == 'RELIANCE'"

In [18]:
def scan_summary(row_filter: str) -> dict[str, int | float]:
    started = time.perf_counter()
    tasks = list(table.scan(row_filter=row_filter, selected_fields=COLUMNS).plan_files())
    elapsed = time.perf_counter() - started

    files = [task.file for task in tasks]
    return {
        "planned_files": len(files),
        "manifest_rows": sum(file.record_count or 0 for file in files),
        "compressed_mb": round(sum(file.file_size_in_bytes or 0 for file in files) / 1024 / 1024, 2),
        "plan_seconds": round(elapsed, 2),
    }


scan_summary(row_filter)

{'planned_files': 11,
 'manifest_rows': 970070,
 'compressed_mb': 17.15,
 'plan_seconds': 0.92}

In [19]:
started = time.perf_counter()
arrow_table = table.scan(row_filter=row_filter, selected_fields=COLUMNS).to_arrow()
df = arrow_table.to_pandas()
elapsed = time.perf_counter() - started

df.shape, round(elapsed, 2)

((970070, 8), 6.26)

In [14]:
df.head(20)

,datetime,trade_date,nse_symbol,open,high,low,close,volume
0,2026-02-12 09:07:00,2026-02-12,RELIANCE,1470.0,1470.0,1470.0,1470.0,0
1,2026-02-12 09:15:00,2026-02-12,RELIANCE,1468.7,1473.0,1465.1,1470.3,115567
2,2026-02-12 09:16:00,2026-02-12,RELIANCE,1470.1,1470.1,1466.2,1466.4,43582
3,2026-02-12 09:17:00,2026-02-12,RELIANCE,1466.4,1467.0,1466.1,1467.0,19965
4,2026-02-12 09:18:00,2026-02-12,RELIANCE,1467.5,1467.5,1466.5,1467.0,23204
5,2026-02-12 09:19:00,2026-02-12,RELIANCE,1466.1,1466.6,1465.3,1465.3,6454
6,2026-02-12 09:20:00,2026-02-12,RELIANCE,1465.3,1466.2,1464.6,1465.6,36847
7,2026-02-12 09:21:00,2026-02-12,RELIANCE,1466.0,1466.6,1465.4,1466.6,19031
8,2026-02-12 09:22:00,2026-02-12,RELIANCE,1466.6,1467.3,1466.2,1467.0,18186
9,2026-02-12 09:23:00,2026-02-12,RELIANCE,1467.0,1467.4,1465.5,1465.9,11981
